In [324]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, asin, sqrt
import json
from collections import defaultdict
import plotly.express as px

In [325]:
pd.set_option('display.max_columns', None)

## Einlesen der Daten

In [326]:
soll = pd.read_csv('Soll-Daten.csv', dtype=str)
soll.head()

,TOURNR,LKW_KENNZ,FAHRERNAME,TOUR,ANKUNFT,ABFAHRT,GEOX,GEOY,STOPP_DAUER,DATUM
0,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 06:00:00,2026-03-02 06:30:00,9.98632,53.52348,0 days 00:30:00,2026-03-02
1,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 06:49:00,2026-03-02 07:19:00,9.99383,53.54751,0 days 00:30:00,2026-03-02
2,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 07:41:00,2026-03-02 08:11:00,9.98632,53.52348,0 days 00:30:00,2026-03-02
3,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 08:31:00,2026-03-02 09:01:00,9.99383,53.54751,0 days 00:30:00,2026-03-02
4,H/42375,OD-TS8050,"Szmajdewicz, Marcin",H/42375,2026-03-02 06:00:00,2026-03-02 06:30:00,9.98632,53.52348,0 days 00:30:00,2026-03-02


In [327]:
ist = pd.read_csv('Ist-Daten.csv', dtype=str)
ist.head()

,Fahrer PIN,Fahrername,Formularnummer,Richtung,Geschwindigkeit,KM-Stand,Gesamtverbrauch,Tankfüllstand,Breitengrad,Längengrad,Position,Fahrzeug,DATUM,KENNZ_CLEAN,TOURNR,Abfahrt,Ankunft
0,8160.0,"Jorczik, Christian",1650,0,0,317003.22,0.0,0.0,"53,6677","9,99287","DE - 22419 Hamburg (Langenhorn), Essener Straß...",OD-TS 8041 - Fahrzeugdetails,2026-03-30,OD-TS8041,H/42647,2026-03-30 06:17:22,2026-03-30 05:18:19
1,8160.0,"Jorczik, Christian",1650,131,0,317021.295,0.0,0.0,"53,55026","9,99017","DE - 20457 Hamburg (Hamburg-Mitte), Alter Wall 32",OD-TS 8041 - Fahrzeugdetails,2026-03-30,OD-TS8041,H/42647,2026-03-30 06:59:55,2026-03-30 06:59:54
2,8160.0,"Jorczik, Christian",1650,0,0,317028.945,0.0,0.0,"53,57519","9,9127","DE - 22525 Hamburg (Bahrenfeld), Winsbergring 15",OD-TS 8041 - Fahrzeugdetails,2026-03-30,OD-TS8041,H/42647,2026-03-30 07:28:24,2026-03-30 07:28:23
3,8160.0,"Jorczik, Christian",1650,0,0,317066.35,0.0,0.0,"53,61828","10,23092","DE - 22145 Braak, Waldweg 2",OD-TS 8041 - Fahrzeugdetails,2026-03-30,OD-TS8041,H/42647,2026-03-30 08:37:25,2026-03-30 08:37:24
4,8160.0,"Jorczik, Christian",1650,0,0,317080.55,0.0,0.0,"53,67164","10,23997","DE - 22926 Ahrensburg, Manhagener Allee 10",OD-TS 8041 - Fahrzeugdetails,2026-03-30,OD-TS8041,H/42647,2026-03-30 09:24:07,2026-03-30 09:24:06


## Unnötige Spalte `TOUR` entfernen in soll

In [328]:
gleich = soll[soll.TOURNR != soll.TOUR]
gleich

,TOURNR,LKW_KENNZ,FAHRERNAME,TOUR,ANKUNFT,ABFAHRT,GEOX,GEOY,STOPP_DAUER,DATUM


In [329]:
soll = soll.drop('TOUR', axis=1)

## Touren mit gleicher Reihenfolge an Stopps filtern
* Für jede Tour wird aus den Soll-Daten eine geordnete Liste von Stopps gebaut und aus den Ist-Daten eine geordnete Liste von GPS-Punkten
* Dann läuft die Funktion toursequencematchesstrict(...) über jeden Ist-Punkt und sucht den nächstgelegenen Soll-Stopp
* Match zählt nur, wenn der Abstand höchstens 300 Meter beträgt
* Es muss mindestens 80 Prozent Abdeckung bestehen, damit die Touren als "gleich" gelten
### Hilfsfunktionen für Geofencing

In [330]:
def haversine(lat1, lon1, lat2, lon2):
    """Berechnet den kürzesten Abstand zwischen zwei Punkten auf einer Kugeloberfläche."""
    # Erdradius in Metern
    R = 6371000
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return 2 * R * asin(sqrt(a))

def parse_decimal_coord(x):
    """Konvertiert eine Koordinate, die als String mit möglichen Tausendertrennzeichen und Komma als Dezimaltrennzeichen vorliegen kann, in einen float-Wert."""
    # komma zu punkt in dezimalzahl
    if pd.isna(x):
        return np.nan
    s = str(x).replace(' ', '').replace('"','')
    s = s.replace(',', '.') if (',' in s and '.' not in s) else s
    return float(s)

MATCH_RADIUS_M = 300  # max Entfernung zwischen Punkten, damit sie als Match gelten (Geofencing-Radius)

### Vorverarbeitung Soll-Daten

In [331]:
# Erwartete Spalten: TOURNR,GEOX,GEOY,ANKUNFT
soll['GEOX'] = soll['GEOX'].astype(str).apply(parse_decimal_coord)
soll['GEOY'] = soll['GEOY'].astype(str).apply(parse_decimal_coord)
soll['ANKUNFT'] = pd.to_datetime(soll['ANKUNFT'], errors='coerce')
soll = soll.sort_values(['TOURNR','ANKUNFT']).reset_index(drop=True)

# Erzeuge für jede TOURNR eine Liste von (lat,lon)
soll_groups = {}
for tournr, g in soll.groupby('TOURNR'):
    coords = list(zip(g['GEOY'].astype(float).tolist(), g['GEOX'].astype(float).tolist()))
    soll_groups[tournr] = coords

### Vorverarbeitung Ist-Daten

In [332]:
# Erwartete Spalten: TOURNR,Breitengrad,Längengrad,Ankunft (oder Abfahrt)
# Normiere Spaltennamen (in Dateien sind viele Varianten); wir nutzen 'TOURNR','Breitengrad','Längengrad','Ankunft'
ist_cols = {c.lower(): c for c in ist.columns}
# Mögliche Spaltennamen: 'Breitengrad' oder 'BREITENGRAD' etc. Wir finden passende.
def find_col(names):
    for n in names:
        if n.lower() in ist_cols:
            return ist_cols[n.lower()]
    return None

col_lat = find_col(['Breitengrad','breitengrad','BREITENGRAD','LAT','LATITUDE','B_LAT'])
col_lon = find_col(['Längengrad','längengrad','LÄNGENGRAD','LON','LONGITUDE','B_LON'])
col_time = find_col(['Ankunft','ankunft','ANREISE','Ankunftszeit','Abfahrt','abfahrt'])
col_tournr = find_col(['TOURNR','tournr','Tournummer','TOUR','TOURNR'])

if col_lat is None or col_lon is None or col_tournr is None:
    raise ValueError("Konnte benötigte Spalten in Ist-Daten nicht finden. Gefundene Spalten: " + ", ".join(ist.columns))

ist['lat'] = ist[col_lat].apply(parse_decimal_coord)
ist['lon'] = ist[col_lon].apply(parse_decimal_coord)
ist[col_time] = pd.to_datetime(ist[col_time], errors='coerce')
ist = ist.sort_values([col_tournr, col_time]).reset_index(drop=True)

# Gruppiere Ist pro TOURNR
ist_groups = {}
for tournr, g in ist.groupby(col_tournr):
    coords = list(zip(g['lat'].astype(float).tolist(), g['lon'].astype(float).tolist()))
    ist_groups[tournr] = coords

### Finde Touren, die matchen


In [333]:
def tour_sequence_matches_strict(
    istseq,
    sollseq,
    radiusm=MATCH_RADIUS_M,
    min_match_ratio=0.8,
    require_unique_soll=False,
    allow_skips=True,
):
    """
    Prüft, ob Ist dieselbe Stop-Reihenfolge wie Soll abfährt.

    - Pro Ist-Punkt wird der nächste Soll-Punkt innerhalb radius gesucht.
    - Die Folge der gematchten Soll-Indizes muss streng aufsteigend sein
      (keine Wiederholung desselben Soll-Stopps).
    - Mindestanteil gematchter Soll-Stopps über min_match_ratio.
    """

    # liste für jeden Ist-Punkt mit: welcher Ist-Punkt das ist, welcher Soll-Index dazu passt und wie groß die Distanz war
    matched = []

    # schleife über alle ist-punkte in reihenfolge
    for i, (ilat, ilon) in enumerate(istseq):
        best_idx = None
        best_dist = float("inf")

        # schleife über alle soll-punkte
        # für jeden punkt wird distanz zum ist-punkt berechnet
        for j, (slat, slon) in enumerate(sollseq):
            d = haversine(ilat, ilon, slat, slon)
            # wenn neu berechnete distanz kleiner als bisher beste distanz -> neuer soll stopp als bester kandidat
            if d < best_dist:
                best_dist = d
                best_idx = j
        # ist bester soll-stop nah genug -> match
        if best_dist <= radiusm:
            matched.append({
                "ist_idx": i,
                "soll_idx": best_idx,
                "dist_m": best_dist
            })
        else:
            matched.append({
                "ist_idx": i,
                "soll_idx": None,
                "dist_m": None
            })

    # filtern alle ungültigen matches heraus 
    # es bleiben nur ist-punkte übrig, die wirklich in dr Nähe eines Soll-Stopps lagen
    valid = [m for m in matched if m["soll_idx"] is not None]
    matched_indices = [m["soll_idx"] for m in valid]

    if len(valid) == 0:
        return False, {
            "matched": matched,
            "matched_count": 0,
            "soll_count": len(sollseq),
            "ist_count": len(istseq),
            "coverage_ratio": 0.0,
            "strictly_increasing": False,
            "unique_soll_count": 0,
            "matched_indices": [],
            "reason": "kein Punkt innerhalb Radius"
        }
    # prüfen ob die Reihenfolge aufsteigend ist
    strictly_increasing = all(
        a < b for a, b in zip(matched_indices, matched_indices[1:])
    )

    unique_soll_count = len(set(matched_indices))
    coverage_ratio = unique_soll_count / len(sollseq) if len(sollseq) else 0.0

    if require_unique_soll:
        unique_ok = len(matched_indices) == unique_soll_count
    else:
        unique_ok = True

    if allow_skips:
        coverage_ok = coverage_ratio >= min_match_ratio
    else:
        coverage_ok = unique_soll_count == len(sollseq)

    matches = strictly_increasing and unique_ok and coverage_ok

    return matches, {
        "matched": matched,
        "matched_count": len(valid),
        "soll_count": len(sollseq),
        "ist_count": len(istseq),
        "coverage_ratio": coverage_ratio,
        "strictly_increasing": strictly_increasing,
        "unique_soll_count": unique_soll_count,
        "matched_indices": matched_indices,
        "reason": None if matches else {
            "strictly_increasing": strictly_increasing,
            "unique_ok": unique_ok,
            "coverage_ok": coverage_ok
        }
    }


# Prüfe alle TOURNR, die in beiden Sets vorkommen 
commontournr = set(soll_groups.keys()).intersection(set(ist_groups.keys()))
matchingtournr = []
matchdetails = {}

for t in sorted(commontournr):
    istseq = ist_groups[t]
    sollseq = soll_groups[t]

    matches, details = tour_sequence_matches_strict(
        istseq,
        sollseq,
        radiusm=MATCH_RADIUS_M,
        min_match_ratio=0.8,
        require_unique_soll=False,
        allow_skips=True
    )

    matchdetails[t] = details

    if matches:
        matchingtournr.append(t)


# Join der DataFrames für gematchte TOURNR 
matchedsoll = soll[soll["TOURNR"].isin(matchingtournr)].copy()
matchedist = ist[ist[col_tournr].isin(matchingtournr)].copy()

# Optional: nummeriere Stop-Positionen innerhalb jeder Tour
matchedsoll["sollstopidx"] = matchedsoll.groupby("TOURNR").cumcount() + 1
matchedist["iststopidx"] = matchedist.groupby(col_tournr).cumcount() + 1

# Einfacher Join auf TOURNR
merged = pd.merge(
    matchedist,
    matchedsoll,
    left_on=col_tournr,
    right_on="TOURNR",
    suffixes=("_ist", "_soll")
)

# Speichere Ergebnisse
pd.DataFrame({"matchingtournr": list(matchingtournr)}).to_csv("matchingtournr.csv", index=False)
matchedsoll.to_csv("matchedsoll.csv", index=False)
matchedist.to_csv("matchedist.csv", index=False)
merged.to_csv("mergedmatchedtours.csv", index=False)

# Speichere Detail-Report
with open("matchdetails.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "MATCH_RADIUS_M": MATCH_RADIUS_M,
            "matchingtournr": matchingtournr,
            "details": matchdetails
        },
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    f"Fertig. Gefundene gematchte Touren: {len(matchingtournr)}. "
    "Listen in matchingtournr.csv, mergedmatchedtours.csv gespeichert."
)

Fertig. Gefundene gematchte Touren: 14. Listen in matchingtournr.csv, mergedmatchedtours.csv gespeichert.


### Sauberer Pair-Merge auf Basis der echten Matches
* bauen aus `matchdetails[t]["matched"]` eine echte Match-Tabelle und nehmen dann die laufenden Indizes ist_idx und soll_idx je Tour und merged genau auf diese Paarung statt nur auf TOURNR
* daraus dann pro Match eine Zeile mit Distanz, Ist-Zeit, Soll-Zeit und daraus delay_min
* Matching-Logik kann mehrere Ist-Punkte demselben Soll-Stopp zuordnen, weil pro Ist-Punkt der nächste Soll-Punkt gesucht wird -> paired_best reduziert das auf genau einen besten Kandidaten pro TOURNR + soll_idx, nämlich den mit der kleinsten Distanz, was für Stop-basierte Delay-Auswertungen meist die sauberste Variante ist

In [334]:
# pair_rows = []

# for tournr in matchingtournr:
#     details = matchdetails[tournr]

#     for m in details["matched"]:
#         if m["soll_idx"] is None:
#             continue

#         pair_rows.append({
#             "TOURNR": tournr,
#             "ist_idx": m["ist_idx"],
#             "soll_idx": m["soll_idx"],
#             "dist_m": m["dist_m"]
#         })

# pairs = pd.DataFrame(pair_rows)

# # Falls keine Paare gefunden wurden
# if pairs.empty:
#     raise ValueError("Keine gültigen Match-Paare in matchdetails gefunden.")

# # Eindeutige laufende Indizes je Tour in Originaldaten

# matchedist = ist[ist[col_tournr].isin(matchingtournr)].copy()
# matchedsoll = soll[soll["TOURNR"].isin(matchingtournr)].copy()

# matchedist = matchedist.sort_values([col_tournr, col_time]).reset_index(drop=True)
# matchedsoll = matchedsoll.sort_values(["TOURNR", "ANKUNFT"]).reset_index(drop=True)

# matchedist["ist_idx"] = matchedist.groupby(col_tournr).cumcount()
# matchedsoll["soll_idx"] = matchedsoll.groupby("TOURNR").cumcount()

# # Nur benötigte Spalten für Ist / Soll vorbereiten

# ist_cols_keep = [col_tournr, "ist_idx", col_time, "lat", "lon"]
# for c in ["Abfahrt", "Ankunft", "Position", "Fahrzeug", "KENNZCLEAN", "DATUM"]:
#     if c in matchedist.columns and c not in ist_cols_keep:
#         ist_cols_keep.append(c)

# soll_cols_keep = ["TOURNR", "soll_idx", "ANKUNFT", "ABFAHRT", "GEOY", "GEOX"]
# for c in ["STRASSE", "ORT", "PLZ", "LKWKENNZ", "FAHRERNAME", "DATUM"]:
#     if c in matchedsoll.columns and c not in soll_cols_keep:
#         soll_cols_keep.append(c)

# ist_for_merge = matchedist[ist_cols_keep].copy()
# soll_for_merge = matchedsoll[soll_cols_keep].copy()

# # Umbenennen für Klarheit
# ist_for_merge = ist_for_merge.rename(columns={
#     col_tournr: "TOURNR",
#     col_time: "IstZeit",
#     "Ankunft": "IstAnkunft",
#     "Abfahrt": "IstAbfahrt",
#     "Position": "IstPosition",
#     "Fahrzeug": "IstFahrzeug",
#     "DATUM": "DATUMist"
# })

# soll_for_merge = soll_for_merge.rename(columns={
#     "ANKUNFT": "SollAnkunft",
#     "ABFAHRT": "SollAbfahrt",
#     "GEOY": "SollLat",
#     "GEOX": "SollLon",
#     "STRASSE": "SollStrasse",
#     "ORT": "SollOrt",
#     "PLZ": "SollPLZ",
#     "DATUM": "DATUMsoll"
# })

# # Echter Merge über TOURNR + Indexpaar

# paired = pairs.merge(
#     ist_for_merge,
#     on=["TOURNR", "ist_idx"],
#     how="left"
# ).merge(
#     soll_for_merge,
#     on=["TOURNR", "soll_idx"],
#     how="left"
# )

# # Delay berechnen
# # Falls IstZeit nicht brauchbar ist, nimm bevorzugt IstAnkunft, sonst IstAbfahrt

# if "IstAnkunft" in paired.columns:
#     paired["IstZeitFinal"] = pd.to_datetime(paired["IstAnkunft"], errors="coerce")
# else:
#     paired["IstZeitFinal"] = pd.NaT

# if "IstAbfahrt" in paired.columns:
#     ist_abfahrt_dt = pd.to_datetime(paired["IstAbfahrt"], errors="coerce")
#     paired["IstZeitFinal"] = paired["IstZeitFinal"].fillna(ist_abfahrt_dt)

# paired["SollZeitFinal"] = pd.to_datetime(paired["SollAnkunft"], errors="coerce")

# paired["delay_min"] = (
#     (paired["IstZeitFinal"] - paired["SollZeitFinal"]).dt.total_seconds() / 60.0
# )

# # nur beste/saubere Paare behalten
# # Falls mehrere Ist-Punkte auf denselben Soll-Stopp zeigen,
# # behalte den mit der kleinsten Distanz

# paired_best = (
#     paired.sort_values(["TOURNR", "soll_idx", "dist_m"])
#           .drop_duplicates(subset=["TOURNR", "soll_idx"], keep="first")
#           .sort_values(["TOURNR", "soll_idx"])
#           .reset_index(drop=True)
# )

# # Exporte

# pairs.to_csv("match_pairs_raw.csv", index=False)
# paired.to_csv("matched_pairs_full.csv", index=False)
# paired_best.to_csv("matched_pairs_best.csv", index=False)

# # Optional: Delay-Zusammenfassung pro Tour
# tour_delay_summary = (
#     paired_best.groupby("TOURNR", dropna=False)
#     .agg(
#         n_pairs=("soll_idx", "count"),
#         avg_delay_min=("delay_min", "mean"),
#         median_delay_min=("delay_min", "median"),
#         max_delay_min=("delay_min", "max"),
#         min_delay_min=("delay_min", "min")
#     )
#     .reset_index()
# )

# tour_delay_summary.to_csv("tour_delay_summary.csv", index=False)

# print(
#     f"Fertig. {len(pairs)} Roh-Paare, "
#     f"{len(paired_best)} eindeutige beste Paare gespeichert."
# )

In [335]:
# paired_best.TOURNR.nunique()

In [336]:
# paired_best[paired_best.TOURNR == 'H/42675']

In [337]:
# soll.TOURNR.nunique()

In [338]:
# ist.TOURNR.nunique()

In [339]:
# soll[soll["TOURNR"] == 'H/42446']

In [340]:
# ist[ist[col_tournr] == 'H/42446']

Unerwartetes Ergebnis: Nur 14 Touren mit Stopps in der richtigen Reihenfolge
## Berechnung der Anzahl der Touren, die die richtigen Stopps abfahren
* zuerst Anzahl der Touren bestimmt, die mindestens 80% der Stopps abfahren -> 90 
* bei 100% waren es aber auch 90% 

In [341]:
# Ist-Touren finden, die mindestens ...% der Soll-Stopps überhaupt anfahren
MIN_STOP_COVERAGE = 1

def tour_stops_covered_any_order(istseq, sollseq, radiusm=MATCH_RADIUS_M, min_coverage=MIN_STOP_COVERAGE):
    """
    Prüft, ob eine Ist-Tour mindestens min_coverage der Soll-Stopps räumlich erreicht,
    unabhängig von der Reihenfolge.

    Logik:
    - Für jeden Soll-Stopp wird geprüft, ob es mindestens einen Ist-Punkt innerhalb radiusm gibt.
    - Gezählt werden eindeutige besuchte Soll-Stopps.
    """
    covered_soll_idx = []
    details = []

    for j, (slat, slon) in enumerate(sollseq):
        best_ist_idx = None
        best_dist = float("inf")

        for i, (ilat, ilon) in enumerate(istseq):
            d = haversine(ilat, ilon, slat, slon)
            if d < best_dist:
                best_dist = d
                best_ist_idx = i

        is_covered = best_dist <= radiusm

        details.append({
            "sollidx": j,
            "best_ist_idx": best_ist_idx,
            "best_dist_m": best_dist,
            "covered": is_covered
        })

        if is_covered:
            covered_soll_idx.append(j)

    covered_count = len(covered_soll_idx)
    soll_count = len(sollseq)
    coverage_ratio = covered_count / soll_count if soll_count else 0.0
    matches = coverage_ratio >= min_coverage

    return matches, {
        "covered_soll_idx": covered_soll_idx,
        "covered_count": covered_count,
        "soll_count": soll_count,
        "coverage_ratio": coverage_ratio,
        "radius_m": radiusm,
        "min_coverage": min_coverage,
        "details": details
    }


# Nur Touren prüfen, die in Soll und Ist vorkommen
common_tournr = set(soll_groups.keys()).intersection(set(ist_groups.keys()))

covered_tournr = []
covered_details = {}

for t in sorted(common_tournr):
    istseq = ist_groups[t]
    sollseq = soll_groups[t]

    ok, details = tour_stops_covered_any_order(
        istseq,
        sollseq,
        radiusm=MATCH_RADIUS_M,
        min_coverage=0.8
    )

    covered_details[t] = details

    if ok:
        covered_tournr.append(t)

# Übersichtstabelle
covered_summary = pd.DataFrame([
    {
        "TOURNR": t,
        "covered_count": covered_details[t]["covered_count"],
        "soll_count": covered_details[t]["soll_count"],
        "coverage_ratio": covered_details[t]["coverage_ratio"]
    }
    for t in covered_tournr
]).sort_values(["coverage_ratio", "TOURNR"], ascending=[False, True]).reset_index(drop=True)

# Nur die Ist-Zeilen dieser Touren
ist_touren_mit_mind_80_prozent_stopps = ist[ist[col_tournr].isin(covered_tournr)].copy()


In [342]:
# PRC-Touren prüfen: Sind alle Soll-Stopps abgefahren worden?
PRC_MATCH_RADIUS_M = 200

prc_tour_col = 'TourID' if 'TourID' in prc.columns else 'TOURNR'

if 'Longitude' not in prc.columns or 'Latitude' not in prc.columns:
    raise KeyError("Fehlende Koordinatenspalten in prc. Erwartet 'Longitude' und 'Latitude'.")

prc['lat'] = prc['Latitude'].apply(parse_decimal_coord)
prc['lon'] = prc['Longitude'].apply(parse_decimal_coord)
prc = prc[prc[prc_tour_col].notna()].copy()
prc = prc.sort_values([prc_tour_col, 'Zeitstempel_Fahrzeug']).reset_index(drop=True)

prc_groups = {}
for tournr, g in prc.groupby(prc_tour_col):
    coords = list(zip(g['lat'].astype(float).tolist(), g['lon'].astype(float).tolist()))
    prc_groups[tournr] = coords

common_prc_tournr = set(soll_groups.keys()).intersection(set(prc_groups.keys()))

prc_covered_tournr = []
prc_covered_details = {}
for t in sorted(common_prc_tournr):
    prcseq = prc_groups[t]
    sollseq = soll_groups[t]

    ok, details = tour_stops_covered_any_order(
        prcseq,
        sollseq,
        radiusm=PRC_MATCH_RADIUS_M,
        min_coverage=1.0
    )

    prc_covered_details[t] = details
    if ok:
        prc_covered_tournr.append(t)

prc_covered_summary = pd.DataFrame([
    {
        'TOURNR': t,
        'covered_count': prc_covered_details[t]['covered_count'],
        'soll_count': prc_covered_details[t]['soll_count'],
        'coverage_ratio': prc_covered_details[t]['coverage_ratio']
    }
    for t in prc_covered_tournr
]).sort_values(['coverage_ratio', 'TOURNR'], ascending=[False, True]).reset_index(drop=True)

prc_touren_mit_kompletter_stopabdeckung = prc[prc[prc_tour_col].isin(prc_covered_tournr)].copy()

print('Gemeinsame Touren zwischen Soll und PRC:', len(common_prc_tournr))
print('PRC-Touren mit kompletter Soll-Stopp-Abdeckung:', len(prc_covered_tournr))
print('PRC-Zeilen dieser Touren:', len(prc_touren_mit_kompletter_stopabdeckung))
prc_covered_summary.head()


Gemeinsame Touren zwischen Soll und PRC: 202
PRC-Touren mit kompletter Soll-Stopp-Abdeckung: 85
PRC-Zeilen dieser Touren: 11736


,TOURNR,covered_count,soll_count,coverage_ratio
0,H/42383,12,12,1.0
1,H/42392,2,2,1.0
2,H/42400,12,12,1.0
3,H/42404,3,3,1.0
4,H/42412,12,12,1.0


In [343]:
ist_touren_mit_mind_80_prozent_stopps['TOURNR'].nunique()

90

In [344]:
# Falls dein DataFrame noch nicht existiert, hier anpassen:
df = ist_touren_mit_mind_80_prozent_stopps.copy()

# Spalten robust finden
fahrer_col = next(c for c in ["FAHRERNAME", "Fahrername", "fahrername"] if c in df.columns)
tour_col = next(c for c in ["TOURNR", "Tournummer", "tournr"] if c in df.columns)

# Pro Fahrer zählen, wie viele eindeutige Touren im DataFrame sind
driver_counts = (
    df[[fahrer_col, tour_col]]
    .drop_duplicates()
    .groupby(fahrer_col, as_index=False)
    .agg(anzahl_touren=(tour_col, "nunique"))
    .sort_values("anzahl_touren", ascending=False)
    .reset_index(drop=True)
)

# Optional: nur Top 15 anzeigen

fig = px.bar(
    driver_counts,
    x="anzahl_touren",
    y=fahrer_col,
    orientation="h",
    text="anzahl_touren",
    title=f"Fahrer mit den meisten Touren mit kompletter Stopp-Abdeckung",
)

fig.update_traces(textposition="outside", cliponaxis=False)
fig.update_layout(
    xaxis_title="Anzahl Touren",
    yaxis_title="Fahrer",
    yaxis=dict(autorange="reversed"),
)

fig.show()

## Gleiche Analyse mit PRC Daten

In [345]:
prc = pd.read_csv('istdaten_prc_clean.csv')

In [346]:
prc.LKW_KENNZ.to_string()

'0        PLO-TS-859\n1        PLO-TS-856\n2        OD-TS-8050\n3        OD-TS-8046\n4        PLO-TS-856\n5        PLO-TS-859\n6        OD-TS-8050\n7        OD-TS-8046\n8        OD-TS-8046\n9        OD-TS-8046\n10       OD-TS-8046\n11       OD-TS-8046\n12       OD-TS-8046\n13       PLO-TS-856\n14       PLO-TS-856\n15       PLO-TS-856\n16       PLO-TS-859\n17       PLO-TS-859\n18       PLO-TS-859\n19       PLO-TS-856\n20       OD-TS-8046\n21       PLO-TS-859\n22       PLO-TS-856\n23       PLO-TS-859\n24       OD-TS-8046\n25       PLO-TS-853\n26       PLO-TS-853\n27       PLO-TS-853\n28       PLO-TS-853\n29       PLO-TS-853\n30       PLO-TS-853\n31       PLO-TS-853\n32       PLO-TS-853\n33       PLO-TS-853\n34       PLO-TS-853\n35       PLO-TS-853\n36       PLO-TS-853\n37       PLO-TS-853\n38       PLO-TS-853\n39       PLO-TS-853\n40       PLO-TS-853\n41       PLO-TS-853\n42       PLO-TS-853\n43       PLO-TS-853\n44       OD-TS-8044\n45       OD-TS-8044\n46       OD-TS-8044\n47       OD-

In [347]:
soll

,TOURNR,LKW_KENNZ,FAHRERNAME,ANKUNFT,ABFAHRT,GEOX,GEOY,STOPP_DAUER,DATUM
0,H/42374,Plo-TS856,"Hesse, Sven-Erik",2026-03-02 06:00:00,2026-03-02 06:30:00,9.98632,53.52348,0 days 00:30:00,2026-03-02
1,H/42374,Plo-TS856,"Hesse, Sven-Erik",2026-03-02 06:49:00,2026-03-02 07:19:00,9.99383,53.54751,0 days 00:30:00,2026-03-02
2,H/42374,Plo-TS856,"Hesse, Sven-Erik",2026-03-02 07:41:00,2026-03-02 08:11:00,9.98632,53.52348,0 days 00:30:00,2026-03-02
3,H/42374,Plo-TS856,"Hesse, Sven-Erik",2026-03-02 08:31:00,2026-03-02 09:01:00,9.99383,53.54751,0 days 00:30:00,2026-03-02
4,H/42375,OD-TS8050,"Szmajdewicz, Marcin",2026-03-02 06:00:00,2026-03-02 06:30:00,9.98632,53.52348,0 days 00:30:00,2026-03-02
...,...,...,...,...,...,...,...,...,...
1615,H/42754,WL-PL431,"Sowa, Mariusz",2026-04-07 09:05:00,2026-04-07 09:35:00,9.98213,53.53497,0 days 00:30:00,2026-04-07
1616,H/42758,PI-EN 444,"Teme, Abdoulaye",2026-04-07 06:00:00,2026-04-07 06:00:00,9.98632,53.52348,0 days 00:00:00,2026-04-07
1617,H/42758,PI-EN 444,"Teme, Abdoulaye",2026-04-07 06:57:00,2026-04-07 07:27:00,10.09782,53.53610,0 days 00:30:00,2026-04-07
1618,H/42758,PI-EN 444,"Teme, Abdoulaye",2026-04-07 07:53:00,2026-04-07 08:23:00,9.98632,53.52348,0 days 00:30:00,2026-04-07


## Kennzeichen normalisieren
Kennzeichen sind in Soll-Daten und PRC-Daten nicht in derselben Form angegeben, wenn es sich um dasselbe Kennzeichen handelt

In [348]:
prc.LKW_KENNZ.unique()

array(['PLO-TS-859', 'PLO-TS-856', 'OD-TS-8050', 'OD-TS-8046',
       'PLO-TS-853', 'OD-TS-8044', 'OD-TS-8048', 'OD-TS-8041',
       'PI-EN-444', 'PI-EN-900', 'WL-PL-431', 'PLO-TS-857'], dtype=object)

In [349]:
soll.LKW_KENNZ.unique()

array(['Plo-TS856', 'OD-TS8050', 'Plo-TS859', 'OD-TS8041', 'OD-TS8044',
       'Plö-TS853', 'OD-TS8046', 'OD-TS8048', 'PI-EN 444', 'RW435',
       'PI-EN 900', 'WL-PL431', 'Leihwagen', 'Plö-TS857'], dtype=object)

In [350]:
import unicodedata

# Funktion zum Normalisieren von Kennzeichen (entfernt "-", Leerzeichen, normalisiert Case und Umlaute)
def normalise_kennzeichen(kz):
    if pd.isna(kz):
        return kz
    kz_str = str(kz)
    # Entferne "-" und Leerzeichen
    kz_normalized = kz_str.replace("-", "").replace(" ", "")
    # Normalisiere zu lowercase für Vergleich
    kz_normalized = kz_normalized.lower()
    # Umlaute normalisieren (ö -> o)
    kz_normalized = unicodedata.normalize('NFKD', kz_normalized)
    kz_normalized = ''.join(c for c in kz_normalized if not unicodedata.combining(c))
    return kz_normalized.lower()

# Erstelle Mapping von normalisiert -> Original aus soll
soll_kennzeichen = soll['LKW_KENNZ'].unique()
mapping = {}
for kz in soll_kennzeichen:
    normalised = normalise_kennzeichen(kz)
    mapping[normalised] = kz  # Original aus soll speichern

# Ersetze Kennzeichen in prc, wenn sie im Mapping gefunden werden
def replace_kennzeichen(kz):
    normalised = normalise_kennzeichen(kz)
    return mapping.get(normalised, kz)  # Rückgabe Original aus soll, wenn Match

prc['LKW_KENNZ'] = prc['LKW_KENNZ'].apply(replace_kennzeichen)

In [351]:
prc.LKW_KENNZ.unique()

array(['Plo-TS859', 'Plo-TS856', 'OD-TS8050', 'OD-TS8046', 'Plö-TS853',
       'OD-TS8044', 'OD-TS8048', 'OD-TS8041', 'PI-EN 444', 'PI-EN 900',
       'WL-PL431', 'Plö-TS857'], dtype=object)

In [352]:
soll.LKW_KENNZ.unique()

array(['Plo-TS856', 'OD-TS8050', 'Plo-TS859', 'OD-TS8041', 'OD-TS8044',
       'Plö-TS853', 'OD-TS8046', 'OD-TS8048', 'PI-EN 444', 'RW435',
       'PI-EN 900', 'WL-PL431', 'Leihwagen', 'Plö-TS857'], dtype=object)

## Tournummern in PRC Daten zuordnen
* neue Spalte `TourID` erstellen mit ergänzten Tournummern aus `soll`
* ursprüngliche Spalte `TOURNR` bleibt

In [353]:
# Datum aus dem Zeitstempel in prc ableiten

prc['DATUM'] = pd.to_datetime(prc['Zeitstempel_Fahrzeug'], errors='coerce').dt.strftime('%Y-%m-%d')

# Eindeutiges Mapping erstellen (eine Tournr pro Kennz+Datum)
if not {'LKW_KENNZ', 'DATUM', 'TOURNR'}.issubset(soll.columns):
    raise KeyError(f"Fehlende Spalten in soll: {set(['LKW_KENNZ', 'DATUM', 'TOURNR']) - set(soll.columns)}")

if not {'LKW_KENNZ', 'DATUM'}.issubset(prc.columns):
    raise KeyError(f"Fehlende Spalten in prc: {set(['LKW_KENNZ', 'DATUM']) - set(prc.columns)}")

tour_mapping = soll.groupby(['LKW_KENNZ', 'DATUM'])['TOURNR'].apply(lambda x: x.iloc[0]).reset_index()
tour_mapping.columns = ['LKW_KENNZ', 'DATUM', 'TOURNR']

# Merge auf prc
prc = prc.merge(tour_mapping, on=['LKW_KENNZ', 'DATUM'], how='left')

# TourID nur ergänzen, wenn sie noch fehlt
if 'TourID' in prc.columns:
    prc['TourID'] = prc['TourID'].fillna(prc['TOURNR'])
else:
    prc['TourID'] = prc['TOURNR']

print(prc['TourID'].isna().sum(), "Zeilen in prc ohne TourID")
print(prc['TourID'].notna().sum(), "Zeilen in prc mit TourID")

403 Zeilen in prc ohne TourID
29027 Zeilen in prc mit TourID


## Entfernen der Zeilen ohne Tournummer

In [354]:
prc = prc[prc.TourID.notna()]

In [355]:
prc

,msg_typ,PositionsID,Longitude,Latitude,Geschwindigkeit,KMStand,Zeitstempel_GPS,Zeitstempel_Fahrzeug,Zeitstempel_Server,LKW_KENNZ,Status,TourID,StationID,SendposID,Ereignis_Typ,DATUM,TOURNR
9,Tourstatus,16053897222,9.98927,53.52262,6,461200,2026-03-01 14:13:25,2026-03-01 14:13:25,2026-03-01 14:17:07,OD-TS8046,22.0,H/42380,NaN,NaN,NaN,2026-03-01,NaN
14,Tourstatus,16053922721,9.98546,53.52336,0,339687,2026-03-01 14:33:05,2026-03-01 14:33:05,2026-03-01 14:33:15,Plo-TS856,22.0,H/42374,NaN,NaN,NaN,2026-03-01,NaN
17,Tourstatus,16053924051,9.98616,53.52343,0,502579,2026-03-01 14:34:07,2026-03-01 14:34:08,2026-03-01 14:34:16,Plo-TS859,22.0,H/42376,NaN,NaN,NaN,2026-03-01,NaN
22,Position,16054978711,9.98546,53.52335,0,339687,2026-03-01 23:59:59,2026-03-02 00:00:00,2026-03-02 00:00:47,Plo-TS856,NaN,H/42374,NaN,NaN,NaN,2026-03-02,H/42374
23,Position,16054980543,9.98616,53.52342,0,502579,2026-03-02 00:00:00,2026-03-02 00:00:00,2026-03-02 00:00:47,Plo-TS859,NaN,H/42376,NaN,NaN,NaN,2026-03-02,H/42376
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29425,Stationsstatus,16264664512,9.95680,53.51534,0,179806,2026-03-31 13:23:34,2026-03-31 13:23:34,2026-03-31 13:24:03,PI-EN 444,11.0,H/42664,H/42664_2-TP12_2,NaN,NaN,2026-03-31,H/42664
29426,FMSStatus,16264664512,9.95680,53.51534,0,179806,2026-03-31 13:23:34,2026-03-31 13:23:34,2026-03-31 13:24:03,PI-EN 444,NaN,H/42664,NaN,NaN,NaN,2026-03-31,H/42664
29427,Position,16264666908,9.89029,53.51766,39,464890,2026-03-31 13:20:10,2026-03-31 13:20:10,2026-03-31 13:24:03,OD-TS8046,NaN,H/42666,NaN,NaN,NaN,2026-03-31,H/42666
29428,Position,16264682099,9.95738,53.51406,25,179806,2026-03-31 13:25:18,2026-03-31 13:25:18,2026-03-31 13:25:38,PI-EN 444,NaN,H/42664,NaN,NaN,NaN,2026-03-31,H/42664


## Touren, die alle Stopps von `soll` abgefahren sind

In [356]:
PRCMATCHRADIUSM = 300

prctourcol = 'TourID' if 'TourID' in prc.columns else 'TOURNR'
if 'Longitude' not in prc.columns or 'Latitude' not in prc.columns:
    raise KeyError('Fehlende Koordinatenspalten in prc. Erwartet Longitude und Latitude.')

prclat = prc['Latitude'].apply(parse_decimal_coord)
prclon = prc['Longitude'].apply(parse_decimal_coord)

prc = prc[prc[prctourcol].notna()].copy()
prc = prc.sort_values([prctourcol, 'Zeitstempel_Fahrzeug']).reset_index(drop=True)

prcgroups = {}
for tournr, g in prc.groupby(prctourcol):
    coords = list(zip(g['Latitude'].astype(float).tolist(), g['Longitude'].astype(float).tolist()))
    prcgroups[tournr] = coords

commonprctournr = set(soll_groups.keys()).intersection(set(prcgroups.keys()))

prccoveredtournr = []
prccovereddetails = {}

for t in sorted(commonprctournr):
    prcseq = prcgroups[t]
    sollseq = soll_groups[t]
    ok, details = tour_stops_covered_any_order(
        prcseq, sollseq,
        radiusm=PRCMATCHRADIUSM,
        min_coverage=1.0,
    )
    prccovereddetails[t] = details
    if ok:
        prccoveredtournr.append(t)

prccoveredsummary = pd.DataFrame([
    {
        'TOURNR': t,
        'covered_count': prccovereddetails[t]['covered_count'],
        'soll_count': prccovereddetails[t]['soll_count'],
        'coverage_ratio': prccovereddetails[t]['coverage_ratio'],
    }
    for t in prccoveredtournr
]).sort_values(['coverage_ratio', 'TOURNR'], ascending=[False, True]).reset_index(drop=True)

prc_komplette_stopabdeckung = prc[prc[prctourcol].isin(prccoveredtournr)].copy()

In [357]:
prc_komplette_stopabdeckung.TourID.nunique()

92

### Ergebnis: 92 in PRC Daten vs. 90 in Soll Daten in Tournummern